# Part 0 — Conceptual Foundation

*Written for idtrack 0.1.2.*

This part builds the mental model you need **before touching code**. It is written for wet-lab researchers and bioinformaticians who want identifier conversion across Ensembl releases and external databases that they can reproduce and check.

**Learning objectives**
- Understand what *identifier drift* is and why it breaks analyses.
- Understand the two axes: **time** (the Ensembl release) and **space** (the database an identifier belongs to).
- Understand what a graph's **snapshot release** is and why it makes results reproducible.
- Know how to read an outcome: **1→0**, **1→1**, **1→n**.

> **Tip:** If you remember one sentence: *idtrack follows each identifier through Ensembl's own history, release by release, to the gene it names at the release you choose, and from there to the database you choose, inside a graph that ends at a fixed release, so every answer can be reproduced and explained.*

## 0.1 — What is idtrack?

In modern biology you rarely analyze a single dataset in isolation. You compare:
- older and newer studies (different Ensembl releases)
- different technologies (bulk and single-cell RNA-seq)
- different annotation sources (Ensembl ids, HGNC symbols, RefSeq, UniProt)

The challenge: **identifiers change** (*identifier drift*).
- Ensembl stable ids can be *retired*, *merged*, *split*, or given a new version across releases.
- Gene symbols are convenient but can be **ambiguous** (one symbol can name several genes) and change over time.
- External databases provide cross-references, but they overlap and can be many-to-many.

idtrack builds a **graph** of Ensembl's identifier history and of the external ids Ensembl links to its genes, and answers: "Given this identifier, which gene does it name at release X (and, optionally, what is that gene's id in database Y)?"

### When should you use idtrack?

Use it when you need:
- **Reproducibility across time**: "map everything to Ensembl release X and keep that fixed for the project".
- **Explanations**: you want to see *why* a mapping happened, or why it did not (`explain=True` returns the path as data).
- **Mixed inputs**: your lists mix Ensembl ids, symbols, UniProt, RefSeq or NCBI gene ids, of different releases.
- **Honest ambiguity**: you prefer an explicit tie or several candidates to a silent guess.

You might *not* need it if you only map a handful of current ids at Ensembl's newest release and do not care to reproduce the mapping later.

## 0.2 — Architecture and workflow

idtrack is easiest to understand with two axes:

1. **Time axis**: Ensembl releases (e.g. 90, 100, 110, 115).
2. **Space axis**: databases of identifiers (Ensembl gene ids, gene symbols, UniProt accessions, RefSeq ids, ...).

A conversion moves along both:

```text
Your identifier (some database, some release)
        |
        v
  [Read it: match the string to a node of the graph, at a release]
        |
        v
  [History travel: follow Ensembl's records release by release]
        |      where the history leads to no gene, an optional bridge (below)
        v
  [The gene(s) at the target release]
        |
        v
  (optional) [Their id(s) in the final database you asked for]
```

Two terms used throughout:

- **History travel**: the walk along Ensembl's own history of an id (new versions, retirements, merges, splits), with the score Ensembl gives each step.
- **Bridge**: a passage from one gene to another through an identifier that the two genes carried one after the other (an external id that moved from a retired gene to a new one, for example). A bridge is taken only where its checks say it is evidence that one gene became the other (never through a synonym; on one assembly only between overlapping places). It is used only where Ensembl's history leads to no gene, `allow_bridges=False` turns it off, and every answer counts its bridges. The full rules: [How idtrack answers](../concepts.rst).

> **Why this matters:** "convert to release 115" chooses a point on the time axis; "give me HGNC symbols" chooses a database on the space axis. The rest is a search inside one graph, which you can reproduce.

### 0.2.1 — The graph and its snapshot release

A graph holds, for one organism:
- nodes for identifiers (Ensembl ids with their versions, the ids without version, and the external ids of the selected databases);
- edges for their relations (Ensembl's history from release to release, gene to transcript to protein, an external id to the gene Ensembl links it to, at the releases and assemblies at which Ensembl lists the link).

Every graph ends at a chosen release, its **snapshot release** (e.g. 115): it holds every release from the organism's first release in idtrack up to it, and nothing newer. That makes results **reproducible**:
- the same snapshot release and the same externals configuration give the same graph and the same conversions;
- a graph file records the selection of external databases and the organism table it was built from, and loading it for another selection is an error, never a silent substitute.

### 0.2.2 — Ensembl's ids and the external databases

idtrack does *not* put every external database Ensembl knows into a graph. Which ones go in, per form (gene, transcript, translation), assembly and release, is the organism's **externals configuration**, a YAML file. Including everything would make the graph and its tables larger and add many-to-many links that make answers more ambiguous.

**Every supported organism ships its externals configuration** (in the package's `default_config` folder), with a curated set of **selected databases**: the species' symbol source (HGNC for human, MGI for mouse, VGNC for pig and the macaques), NCBI gene ids, RefSeq, UniProt, CCDS, miRBase, RFAM, Havana and clone-based names, and their synonyms. So a newcomer needs no configuration at all:

1. install idtrack (Part 1);
2. build or load a graph (Part 2);
3. convert, and read the result (Part 3).

Choosing your own selection of databases is an advanced topic (Part 6). The shipped selections and how databases are filed: [External databases](../databases.rst).

### 0.2.3 — Reading an outcome (1→0, 1→1, 1→n)

When you convert one identifier, there are three shapes of answer:

- **1→0**: no answer in the database you asked for. The string is not an identifier of the graph (`status == "not_found"`), it carries a version the graph does not hold (`"unknown_version"`), or it leads to no gene at the target release (`"no_conversion"`). `result.reason` names the cause, `API.reason_text(result)` says it in a sentence. (Genes that have no id in the database you asked for are `"no_target"`: the genes themselves stand in, and `API.classify` counts them as "alternative targets" of 1→1 or 1→n.)
- **1→1**: one best answer (`"converted"`).
- **1→n**: several targets in the best answer: several genes equal in all evidence, a **tie** (`"tie"`: idtrack returns them all rather than pick one), or several ids of one gene in the database asked for (two RefSeq accessions of one gene: also a `"tie"`). A 1→1 answer can have other, lower-ranked candidates too: `result.view("all")` lists them.

idtrack tells you which case you are in. That is a feature: it prevents silent mistakes. Statuses, reasons and how candidates are ranked: [How idtrack answers](../concepts.rst).

### 0.2.4 — Genome assemblies are part of the mapping

A **genome assembly** (a "build") is the reference coordinate system on which genes and transcripts are defined. Examples:

- human: **GRCh38** and **GRCh37**
- mouse: **GRCm39** and **GRCm38**
- pig: **Sscrofa11.1** and **Sscrofa10.2**
- rhesus macaque: **Mmul_10**, **Mmul_8.0.1** and **MMUL_1**
- cynomolgus macaque: **Macaca_fascicularis_6.0** and **5.0**

Two datasets can use the same kind of identifiers and still be anchored to different builds, because they were annotated with different GTFs or reference packages: a common situation in atlas building.

A graph holds all assemblies of its releases:

- an external id is linked to a gene for the assemblies and releases at which Ensembl lists the link;
- a conversion follows, at each release, the assembly Ensembl maintained at that release (human: NCBI36 to release 54, GRCh37 to 75, GRCh38 from 76), so identifiers of an older build convert to the build of your target release;
- ids of a build Ensembl kept beside the maintained one (human GRCh37 after release 75) are in the graph as ids to convert from.

Human is the one organism with two assemblies served in parallel (GRCh38 and GRCh37). Mouse, pig and the two macaques have one maintained assembly per release, but their older assemblies still matter for older datasets. Assemblies, organisms and their keys: [Organisms](../organisms.rst).

### 0.2.5 — The names you will use

You do not need to be a developer to use these, but knowing the names helps you follow the tutorials:

1. **`idtrack.API`**: the entry point.
   - `API.build(organism, snapshot_release, local_repository)` builds a graph (or loads its file when the folder holds it); `API.load(path)` loads a graph file and never builds.
   - `api.convert(...)` and `api.convert_many(...)` return `idtrack.results.ConversionResult` objects: every candidate, ranked, the best marked, with the reason when there is none.
2. **`idtrack.build.DatabaseManager`**: reads Ensembl's tables and keeps them in the local repository (needed only to build a graph).
3. **`idtrack.engine.Track`**: the search and ranking in the graph; you use it through the API (`api.engine`).
4. **`idtrack.harmonize.HarmonizeFeatures`**: converts the gene ids of several `.h5ad` datasets to one shared set (the `harmonize` extra; Part 4).
5. **`idtrack.organisms`, `idtrack.settings`, `idtrack.errors`**: the organism table, the settings (search budget, how Ensembl is reached) and the errors you can catch (each an `idtrack.errors.IdtrackError`).

The command line (`idtrack build`, `idtrack convert`, ...) does the same without Python: [Command line](../cli.rst).

## 0.3 — Assumptions and limitations

idtrack assumes you have:
- a writable **local repository**: a folder of your choice that holds the downloaded tables and the graph files (and, if you make one, your own externals configuration);
- network access the first time you build a graph (Ensembl REST and Ensembl's tables, read over MySQL or HTTPS); loading a graph that is on disk needs none;
- disk space for the tables a graph is built from: from 0.24 GB (cynomolgus) to at most 9.3 GB (human). Times, sizes and memory: [Time and memory](../performance.rst).

idtrack can:
- follow identifiers across Ensembl releases up to the snapshot release, on all assemblies of the organism;
- give the id of the gene reached in any selected database;
- show ambiguity instead of hiding it, and return genes that are equal in all evidence as a tie.

idtrack cannot remove ambiguity that is in the data:
- if Ensembl's history says a gene split into several, the true answer may be **1→n**;
- a symbol used for several genes can stay ambiguous;
- an answer is as good as Ensembl's records and the links it lists.

Where an answer can be wrong, and how often that was measured: [Limitations and known costs](../limitations.rst).

> **Why this matters:** idtrack prefers to be honestly ambiguous rather than silently wrong. If you need a single answer from a 1→n case, make that choice explicitly, and record how you made it.

In [1]:
# 1) Minimal setup cell (safe to run in any notebook)
from __future__ import annotations

import os
from pathlib import Path

import idtrack
from idtrack import API

# The local repository: a folder of your choice. The tutorials read it from IDTRACK_LOCAL_REPO (a convention of these
# notebooks only; idtrack itself reads no such variable) and fall back to ./idtrack_cache.
LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

API.configure_logger()

print('idtrack version:', idtrack.__version__)
print('Local repository:', LOCAL_REPOSITORY)

idtrack version: 0.1.2
Local repository: idtrack_cache


The cell above does three things:
1. chooses the local repository (the folder of tables and graph files);
2. imports `idtrack.API`;
3. shows idtrack's log messages (downloads, graph builds) on the console.

In [2]:
# 2) Resolve organism names the way idtrack expects (asks Ensembl REST, so it needs the network).
# A name of idtrack's own table also resolves offline: idtrack.organisms.resolve_organism('human').

for query in ['human', 'mus musculus', 'sus scrofa']:
    formal_name, latest_release = API.ensembl_organism(query)
    print(f'{query!r} -> {formal_name!r} (latest Ensembl release: {latest_release})')

2026-11-24 11:51:27 INFO:idtrack.build.species: Ensembl Rest API query to get the organism names and associated releases.


'human' -> 'homo_sapiens' (latest Ensembl release: 116)
'mus musculus' -> 'mus_musculus' (latest Ensembl release: 116)
'sus scrofa' -> 'sus_scrofa' (latest Ensembl release: 116)


Other forms resolve the same way: `'Macaca mulatta'` or the taxon id `9544` give `'macaca_mulatta'`.

Matching is case-insensitive. Species names, taxon ids and the short names `'human'`, `'mouse'`, `'pig'`, `'rhesus'` and `'cynomolgus'` always resolve (`'pig'` is `sus_scrofa`, although Ensembl also lists every pig breed under that common name); another common name shared by several Ensembl species raises an error that lists the candidates. The latest release is the newest one Ensembl serves today; the tutorials fix the snapshot release instead (Part 2), so that their answers do not change when Ensembl publishes a new release.

From now on, the tutorials use the **formal Ensembl names** (`homo_sapiens`, `mus_musculus`, ...).

## 0.4 — Tutorial roadmap

1. **Install and check** (Part 1, [01_installation_guide](01_installation_guide.ipynb)).
2. **Build or load a graph** (Part 2, [02_initialization_graph](02_initialization_graph.ipynb)).
3. **Convert, and read the result**: the human API in depth (Part 3, [03_api_deep_dive_human](03_api_deep_dive_human.ipynb)).
4. **Harmonize datasets** (Part 4, [04_tutorial_harmonization](04_tutorial_harmonization.ipynb); the `harmonize` extra).
5. **Across species**: mouse and pig genes to their human orthologs (Part 5, [05_tutorial_humanization_mouse_pig_to_human](05_tutorial_humanization_mouse_pig_to_human.ipynb)).
6. **Your own selection of external databases** (Part 6, [06_prepare_new_external_yaml](06_prepare_new_external_yaml.ipynb); advanced: the shipped configuration serves Parts 1 to 5).
7. **Advanced topics**: selections of databases per project, troubleshooting, pipelines (Part 7, [07_advanced_topics](07_advanced_topics.ipynb)).
8. **Appendix**: self-tests of a graph ([99_initialization_test](99_initialization_test.ipynb)).

For a ten-minute start without notebooks: [Quickstart](../usage.rst).

## 0.5 — Quick troubleshooting

If something fails, check these first:

1. **Permissions**: is your local repository writable?
2. **Network**: a first build reaches Ensembl REST and reads Ensembl's tables over MySQL (ports 3306, 5306, 3337) where they answer, over HTTPS otherwise. Where the MySQL ports are blocked, set `IDTRACK_ENSEMBL_ACCESS=https` so that they are never tried (the same tables, rows and values); behind a proxy, set `HTTPS_PROXY`.
3. **Disk space**: from 0.24 GB (cynomolgus) to at most 9.3 GB (human) of tables ([Time and memory](../performance.rst)).
4. **Snapshot release**: it must be a release of the organism in Ensembl and in idtrack's organism table; the graph covers every release from the organism's first release in idtrack up to it, on all its assemblies.
5. **Externals configuration**: every supported organism ships one; a `<organism>_externals_modified.yml` in your local repository takes its place (Part 6). If a build warns that releases have no entry in the configuration, extend it (Part 6, section 6.6).

> **Tip:** For a deeper checklist, see section 7.3 of Part 7.